# Pre-processing Pipeline – Spatial & Radiometric Standardization

This notebook takes raw DICOM CT volumes and produces uniform **256 × 256 × 256** 
volumes at **1 mm isotropic** spacing with:
- **Body / ROI masking** to remove air voxels
- **Bone windowing** followed by **CLAHE** contrast enhancement (bimodal separation)
- **Bone-only isolation** via thresholding & morphological operations

Ready for downstream 3D reconstruction and fracture segmentation.

## 1. Imports & Configuration

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import SimpleITK as sitk
import matplotlib.pyplot as plt
from pathlib import Path
from skimage import exposure, morphology
from scipy import ndimage

# ── Paths ──────────────────────────────────────────────────────
PROJECT_ROOT = Path(r"c:\Users\Chan Zheng Shao\OneDrive\Desktop\Github Repo\TestProject\TestProject")
DATA_DIR_LEFT  = PROJECT_ROOT / "data" / "raw" / "fractured" / "PartLeft"
DATA_DIR_RIGHT = PROJECT_ROOT / "data" / "raw" / "fractured" / "PartRight"
OUTPUT_DIR     = PROJECT_ROOT / "data" / "processed" / "fractured"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Pipeline constants ─────────────────────────────────────────
TARGET_SPACING = (1.0, 1.0, 1.0)   # mm – isotropic
TARGET_SHAPE   = (256, 256, 256)   # voxels – final crop
BONE_WINDOW_CENTER = 400           # HU
BONE_WINDOW_WIDTH  = 1800          # HU  →  range [−500, 1300]
MIN_SLICES = 3                     # skip scout / localizer images

# ── Body-masking constants ────────────────────────────────────
BODY_THRESHOLD_HU = -400           # air ≈ −1000 HU; body > −400
BODY_MORPH_RADIUS = 5              # disk radius for closing gaps

# ── CLAHE constants ───────────────────────────────────────────
CLAHE_CLIP_LIMIT  = 0.03           # contrast limit for adaptive eq.
CLAHE_KERNEL_SIZE = None           # None → skimage default (1/8 image)

# ── Bone isolation constants ──────────────────────────────────
BONE_THRESHOLD_HU   = 200          # cortical ≈ 300–3000; trabecular ≈ 150–300
BONE_CLOSE_RADIUS   = 2            # closing radius to fill small gaps
BONE_OPEN_RADIUS    = 1            # opening radius to remove speckle
BONE_MIN_OBJ_SIZE   = 64           # remove small noise objects (voxels)

print("Configuration ready.")
print(f"  Target spacing : {TARGET_SPACING}")
print(f"  Target shape   : {TARGET_SHAPE}")
print(f"  Bone window    : center={BONE_WINDOW_CENTER}, width={BONE_WINDOW_WIDTH}")
print(f"  Body mask thr  : {BODY_THRESHOLD_HU} HU")
print(f"  CLAHE clip     : {CLAHE_CLIP_LIMIT}")
print(f"  Bone iso thr   : {BONE_THRESHOLD_HU} HU")
print(f"  Output dir     : {OUTPUT_DIR}")

## 2. Data Discovery & Filtering

In [ ]:
def get_case_directories(base_dir, part_name):
    """Discover case directories under a given part directory."""
    cases = []
    base = Path(base_dir)
    if base.exists():
        for case_path in sorted(base.iterdir()):
            if case_path.is_dir():
                cases.append({
                    "case_id": case_path.name,
                    "part": part_name,
                    "case_path": str(case_path),
                })
    return cases


def extract_dicom_metadata(case_path):
    """Return (size, spacing) for a DICOM series, or (None, None) on failure."""
    reader = sitk.ImageSeriesReader()
    dicom_names = reader.GetGDCMSeriesFileNames(case_path)
    if not dicom_names:
        return None, None
    reader.SetFileNames(dicom_names)
    try:
        img = reader.Execute()
        return img.GetSize(), img.GetSpacing()
    except Exception as e:
        print(f"  ⚠ Error reading {case_path}: {e}")
        return None, None


# Discover all cases
all_cases = []
all_cases.extend(get_case_directories(DATA_DIR_LEFT,  "PartLeft"))
all_cases.extend(get_case_directories(DATA_DIR_RIGHT, "PartRight"))

df_cases = pd.DataFrame(all_cases)

# Extract metadata
print("Extracting metadata …")
sizes, spacings = [], []
for _, row in df_cases.iterrows():
    sz, sp = extract_dicom_metadata(row["case_path"])
    sizes.append(sz)
    spacings.append(sp)

df_cases["original_size"]    = sizes
df_cases["original_spacing"] = spacings

# Filter out degenerate cases (≤ MIN_SLICES)
df_cases["z_slices"] = df_cases["original_size"].apply(
    lambda s: s[2] if s is not None else 0
)
mask_valid = df_cases["z_slices"] >= MIN_SLICES
df_valid   = df_cases[mask_valid].reset_index(drop=True)
df_skipped = df_cases[~mask_valid]

print(f"\nTotal cases found : {len(df_cases)}")
print(f"Valid cases       : {len(df_valid)}")
print(f"Skipped cases     : {len(df_skipped)}")

if len(df_skipped) > 0:
    print("\nSkipped (too few slices):")
    display(df_skipped[["case_id", "part", "z_slices"]])

print("\nValid cases:")
display(df_valid[["case_id", "part", "original_size", "original_spacing"]])

## 3. DICOM Loading

In [ ]:
def load_dicom_volume(case_path: str) -> sitk.Image:
    """Load a DICOM series as a 3-D SimpleITK image (HU values)."""
    reader = sitk.ImageSeriesReader()
    dicom_names = reader.GetGDCMSeriesFileNames(case_path)
    reader.SetFileNames(dicom_names)
    reader.MetaDataDictionaryArrayUpdateOn()
    reader.LoadPrivateTagsOn()
    image = reader.Execute()
    # SimpleITK automatically applies RescaleSlope / RescaleIntercept
    # so the pixel values are already in Hounsfield Units.
    return image


# Quick sanity check on the first valid case
sample_path = df_valid.iloc[0]["case_path"]
sample_img  = load_dicom_volume(sample_path)
print(f"Sample case : {df_valid.iloc[0]['case_id']}")
print(f"  Size      : {sample_img.GetSize()}")
print(f"  Spacing   : {tuple(round(s, 4) for s in sample_img.GetSpacing())}")
print(f"  Origin    : {tuple(round(o, 2) for o in sample_img.GetOrigin())}")
print(f"  Direction : {sample_img.GetDirection()}")

## 4. Spatial Standardization

1. **Resample** to isotropic 1 mm × 1 mm × 1 mm using **linear interpolation**.  
2. **Center-crop or zero-pad** to a fixed 256 × 256 × 256 volume.

In [ ]:
def resample_to_isotropic(
    image: sitk.Image,
    target_spacing: tuple = TARGET_SPACING,
    interpolator=sitk.sitkLinear,
    default_value: float = -1000.0,   # air in HU
) -> sitk.Image:
    """Resample an image to the target isotropic spacing."""
    original_spacing = image.GetSpacing()
    original_size    = image.GetSize()

    # Compute new grid size so that the physical extent is preserved
    new_size = [
        int(round(osz * osp / tsp))
        for osz, osp, tsp in zip(original_size, original_spacing, target_spacing)
    ]

    resampler = sitk.ResampleImageFilter()
    resampler.SetOutputSpacing(target_spacing)
    resampler.SetSize(new_size)
    resampler.SetOutputDirection(image.GetDirection())
    resampler.SetOutputOrigin(image.GetOrigin())
    resampler.SetTransform(sitk.Transform())
    resampler.SetDefaultPixelValue(default_value)
    resampler.SetInterpolator(interpolator)

    return resampler.Execute(image)


def center_crop_or_pad(
    image: sitk.Image,
    target_shape: tuple = TARGET_SHAPE,
    pad_value: float = -1000.0,   # air in HU
) -> sitk.Image:
    """Center-crop (if larger) or zero-pad (if smaller) to target_shape."""
    arr = sitk.GetArrayFromImage(image)          # (Z, Y, X)
    current_shape = arr.shape                     # numpy order

    result = np.full(target_shape, pad_value, dtype=arr.dtype)  # (Z, Y, X)

    # For each axis: compute source & destination slices
    slices_src = []
    slices_dst = []
    for i in range(3):
        cs = current_shape[i]
        ts = target_shape[i]
        if cs >= ts:
            # Center-crop
            start = (cs - ts) // 2
            slices_src.append(slice(start, start + ts))
            slices_dst.append(slice(0, ts))
        else:
            # Zero-pad
            pad_before = (ts - cs) // 2
            slices_src.append(slice(0, cs))
            slices_dst.append(slice(pad_before, pad_before + cs))

    result[slices_dst[0], slices_dst[1], slices_dst[2]] = (
        arr[slices_src[0], slices_src[1], slices_src[2]]
    )

    # Convert back to SimpleITK, carrying over spacing & direction
    out_image = sitk.GetImageFromArray(result)
    out_image.SetSpacing(image.GetSpacing())
    out_image.SetDirection(image.GetDirection())
    # Adjust origin so that the center of the volume stays the same
    out_image.SetOrigin(image.GetOrigin())
    return out_image


# ── Quick demo ─────────────────────────────────────────────────
resampled = resample_to_isotropic(sample_img)
cropped   = center_crop_or_pad(resampled)

print("Spatial standardization demo:")
print(f"  Original  → size={sample_img.GetSize()}, spacing={tuple(round(s,4) for s in sample_img.GetSpacing())}")
print(f"  Resampled → size={resampled.GetSize()},  spacing={tuple(round(s,4) for s in resampled.GetSpacing())}")
print(f"  Cropped   → size={cropped.GetSize()},  spacing={tuple(round(s,4) for s in cropped.GetSpacing())}")

## 5. Body / ROI Masking

Remove surrounding **air voxels** by:
1. Thresholding the HU volume at `BODY_THRESHOLD_HU` (−400 HU)
2. Morphological **closing** (slice-by-slice) to fill internal gaps
3. Connected-component analysis to keep only the **largest body region**
4. Hole-filling to handle any remaining interior cavities

In [ ]:
def create_body_mask(
    volume_hu: np.ndarray,
    threshold: float = BODY_THRESHOLD_HU,
    morph_radius: int = BODY_MORPH_RADIUS,
) -> np.ndarray:
    """Return a binary body mask (1 = body, 0 = air) for a 3-D HU volume."""
    # Step 1 – coarse threshold
    mask = (volume_hu > threshold).astype(np.uint8)

    # Step 2 – morphological closing slice-by-slice (fill internal gaps)
    struct = morphology.disk(morph_radius)
    for z in range(mask.shape[0]):
        mask[z] = morphology.binary_closing(mask[z], struct).astype(np.uint8)

    # Step 3 – keep only the largest connected component
    labelled, num_features = ndimage.label(mask)
    if num_features > 1:
        sizes = ndimage.sum(mask, labelled, range(1, num_features + 1))
        largest = np.argmax(sizes) + 1
        mask = (labelled == largest).astype(np.uint8)

    # Step 4 – fill remaining interior holes slice-by-slice
    for z in range(mask.shape[0]):
        mask[z] = ndimage.binary_fill_holes(mask[z]).astype(np.uint8)

    return mask


def apply_body_mask(
    volume_hu: np.ndarray,
    mask: np.ndarray,
    fill_value: float = -1000.0,
) -> np.ndarray:
    """Zero-out air voxels in volume using body mask."""
    out = volume_hu.copy()
    out[mask == 0] = fill_value
    return out


# ── Demo: body masking on sample case ──────────────────────────
demo_hu = sitk.GetArrayFromImage(cropped).astype(np.float32)
demo_body_mask = create_body_mask(demo_hu)
demo_masked    = apply_body_mask(demo_hu, demo_body_mask)

mid_z = demo_hu.shape[0] // 2

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(demo_hu[mid_z], cmap="gray", vmin=-500, vmax=1300)
axes[0].set_title("Original HU (axial mid-slice)")
axes[0].axis("off")

axes[1].imshow(demo_body_mask[mid_z], cmap="gray")
axes[1].set_title("Body Mask")
axes[1].axis("off")

axes[2].imshow(demo_masked[mid_z], cmap="gray", vmin=-500, vmax=1300)
axes[2].set_title("Masked Volume (air removed)")
axes[2].axis("off")

plt.suptitle(f"Body / ROI Masking – {df_valid.iloc[0]['case_id']}", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

body_voxels = demo_body_mask.sum()
total_voxels = demo_body_mask.size
print(f"Body voxels : {body_voxels:,} / {total_voxels:,}  ({100*body_voxels/total_voxels:.1f}%)")
print(f"Air removed : {total_voxels - body_voxels:,} voxels")

## 6. Radiometric Standardization – Bone Windowing

Apply a **bone CT window** (center=400 HU, width=1800 HU → HU range [−500, 1300]),  
then normalize to **[0, 1]**.

In [ ]:
def apply_bone_window(
    image_array: np.ndarray,
    center: float = BONE_WINDOW_CENTER,
    width: float  = BONE_WINDOW_WIDTH,
) -> np.ndarray:
    """Apply bone CT windowing and normalize to [0, 1]."""
    lower = center - width / 2.0   # −500
    upper = center + width / 2.0   # 1300

    windowed = np.clip(image_array, lower, upper)
    normalized = (windowed - lower) / (upper - lower)   # → [0, 1]
    return normalized.astype(np.float32)


# Demo
demo_windowed = apply_bone_window(demo_masked)
print(f"Before windowing : min={demo_masked.min():.1f}, max={demo_masked.max():.1f}")
print(f"After  windowing : min={demo_windowed.min():.4f}, max={demo_windowed.max():.4f}")

## 7. CLAHE Enhancement & Bimodal Separation

Apply **Contrast-Limited Adaptive Histogram Equalization (CLAHE)** slice-by-slice  
to the bone-windowed, body-masked volume.  
Then visualize the **histogram** to confirm clear **bimodal separation**  
(soft-tissue peak vs. bone peak).

In [ ]:
def apply_clahe_3d(
    volume_norm: np.ndarray,
    clip_limit: float = CLAHE_CLIP_LIMIT,
    kernel_size=CLAHE_KERNEL_SIZE,
) -> np.ndarray:
    """Apply CLAHE slice-by-slice to a [0,1] normalized volume."""
    out = np.empty_like(volume_norm)
    for z in range(volume_norm.shape[0]):
        out[z] = exposure.equalize_adapthist(
            volume_norm[z],
            clip_limit=clip_limit,
            kernel_size=kernel_size,
        )
    return out.astype(np.float32)


# ── Demo: CLAHE on sample case ─────────────────────────────────
demo_clahe = apply_clahe_3d(demo_windowed)

# --- Before / After axial view ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(demo_windowed[mid_z], cmap="gray", vmin=0, vmax=1)
axes[0].set_title("Before CLAHE (bone-windowed)")
axes[0].axis("off")

axes[1].imshow(demo_clahe[mid_z], cmap="gray", vmin=0, vmax=1)
axes[1].set_title("After CLAHE")
axes[1].axis("off")

plt.suptitle(f"CLAHE Enhancement – {df_valid.iloc[0]['case_id']}", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# --- Bimodal histogram ---
# Only consider voxels inside the body mask
body_vals_before = demo_windowed[demo_body_mask == 1].ravel()
body_vals_after  = demo_clahe[demo_body_mask == 1].ravel()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].hist(body_vals_before, bins=256, color="steelblue", alpha=0.8, edgecolor="none")
axes[0].set_title("Histogram BEFORE CLAHE (body voxels only)")
axes[0].set_xlabel("Normalized Intensity")
axes[0].set_ylabel("Frequency")
axes[0].axvline(x=0.28, color="red", linestyle="--", linewidth=1, label="soft-tissue region")
axes[0].axvline(x=0.55, color="orange", linestyle="--", linewidth=1, label="bone region")
axes[0].legend()

axes[1].hist(body_vals_after, bins=256, color="darkorange", alpha=0.8, edgecolor="none")
axes[1].set_title("Histogram AFTER CLAHE (body voxels only) – Bimodal Separation")
axes[1].set_xlabel("CLAHE-Enhanced Intensity")
axes[1].set_ylabel("Frequency")
axes[1].legend(["CLAHE enhanced"])

plt.suptitle("Bimodal Separation: Soft Tissue vs Bone", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"CLAHE output : min={demo_clahe.min():.4f}, max={demo_clahe.max():.4f}")

## 8. Bone-Only Isolation

Isolate **osseous structures** (cortical + trabecular bone) from the original HU volume via:
1. **Thresholding** at `BONE_THRESHOLD_HU` (200 HU)
2. **Intersection** with body mask to exclude air artefacts
3. **Morphological closing** to bridge small fracture-line gaps
4. **Morphological opening** to remove speckle noise
5. **Small-object removal** to discard residual noise clusters

In [ ]:
def isolate_bone(
    volume_hu: np.ndarray,
    body_mask: np.ndarray,
    threshold: float = BONE_THRESHOLD_HU,
    close_radius: int = BONE_CLOSE_RADIUS,
    open_radius: int = BONE_OPEN_RADIUS,
    min_size: int = BONE_MIN_OBJ_SIZE,
) -> np.ndarray:
    """Return binary bone mask from an HU volume."""
    # Step 1 – threshold
    bone = (volume_hu >= threshold).astype(np.uint8)

    # Step 2 – intersect with body mask
    bone = bone * body_mask

    # Step 3 – morphological closing (fill fracture lines)
    struct_close = morphology.ball(close_radius)
    bone = morphology.binary_closing(bone, struct_close).astype(np.uint8)

    # Step 4 – morphological opening (remove speckle)
    struct_open = morphology.ball(open_radius)
    bone = morphology.binary_opening(bone, struct_open).astype(np.uint8)

    # Step 5 – remove small objects
    bone = morphology.remove_small_objects(
        bone.astype(bool), min_size=min_size
    ).astype(np.uint8)

    return bone


# ── Demo: bone isolation on sample case ────────────────────────
demo_bone_mask = isolate_bone(demo_hu, demo_body_mask)

bone_voxels = demo_bone_mask.sum()
print(f"Bone voxels isolated : {bone_voxels:,} / {total_voxels:,}  ({100*bone_voxels/total_voxels:.1f}%)")

# --- Multi-view visualisation ---
mid_y = demo_hu.shape[1] // 2
mid_x = demo_hu.shape[2] // 2

fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Row 1: Original HU with bone overlay
for ax, slc, title in zip(
    axes[0],
    [demo_hu[mid_z], demo_hu[:, mid_y, :], demo_hu[:, :, mid_x]],
    ["Axial (original)", "Coronal (original)", "Sagittal (original)"],
):
    ax.imshow(slc, cmap="gray", vmin=-500, vmax=1300)
    ax.set_title(title)
    ax.axis("off")

# Row 2: Bone mask overlay
for ax, slc_hu, slc_bone, title in zip(
    axes[1],
    [demo_hu[mid_z], demo_hu[:, mid_y, :], demo_hu[:, :, mid_x]],
    [demo_bone_mask[mid_z], demo_bone_mask[:, mid_y, :], demo_bone_mask[:, :, mid_x]],
    ["Axial (bone isolated)", "Coronal (bone isolated)", "Sagittal (bone isolated)"],
):
    ax.imshow(slc_hu, cmap="gray", vmin=-500, vmax=1300)
    ax.imshow(slc_bone, cmap="Reds", alpha=0.4)
    ax.set_title(title)
    ax.axis("off")

plt.suptitle(f"Bone-Only Isolation – {df_valid.iloc[0]['case_id']}", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 9. Pipeline Execution & Save

In [ ]:
def preprocess_single_case(
    case_path: str,
    case_id: str,
    output_dir: Path,
    target_spacing: tuple = TARGET_SPACING,
    target_shape: tuple   = TARGET_SHAPE,
) -> dict:
    """Full preprocessing for one case.  Returns a summary dict."""
    # 1. Load DICOM
    image = load_dicom_volume(case_path)
    orig_size    = image.GetSize()
    orig_spacing = tuple(round(s, 4) for s in image.GetSpacing())

    # 2. Resample to isotropic spacing
    resampled = resample_to_isotropic(image, target_spacing)

    # 3. Center-crop / pad to target shape
    cropped = center_crop_or_pad(resampled, target_shape)

    # 4. Body / ROI masking
    arr_hu = sitk.GetArrayFromImage(cropped).astype(np.float32)
    body_mask = create_body_mask(arr_hu)
    arr_masked = apply_body_mask(arr_hu, body_mask)

    # 5. Bone windowing → [0, 1]
    arr_windowed = apply_bone_window(arr_masked)

    # 6. CLAHE enhancement
    arr_clahe = apply_clahe_3d(arr_windowed)

    # 7. Bone-only isolation
    bone_mask = isolate_bone(arr_hu, body_mask)

    # 8. Save enhanced volume as NIfTI
    out_image = sitk.GetImageFromArray(arr_clahe)
    out_image.SetSpacing(cropped.GetSpacing())
    out_image.SetDirection(cropped.GetDirection())
    out_image.SetOrigin(cropped.GetOrigin())

    vol_path = output_dir / f"{case_id}.nii.gz"
    sitk.WriteImage(out_image, str(vol_path))

    # 9. Save bone mask as NIfTI
    bone_image = sitk.GetImageFromArray(bone_mask.astype(np.uint8))
    bone_image.SetSpacing(cropped.GetSpacing())
    bone_image.SetDirection(cropped.GetDirection())
    bone_image.SetOrigin(cropped.GetOrigin())

    bone_path = output_dir / f"{case_id}_bone_mask.nii.gz"
    sitk.WriteImage(bone_image, str(bone_path))

    return {
        "case_id": case_id,
        "orig_size": orig_size,
        "orig_spacing": orig_spacing,
        "proc_size": out_image.GetSize(),
        "proc_spacing": tuple(round(s, 4) for s in out_image.GetSpacing()),
        "intensity_min": float(arr_clahe.min()),
        "intensity_max": float(arr_clahe.max()),
        "body_voxel_pct": round(100 * body_mask.sum() / body_mask.size, 1),
        "bone_voxel_pct": round(100 * bone_mask.sum() / bone_mask.size, 1),
        "vol_path": str(vol_path),
        "bone_path": str(bone_path),
    }


# ── Run for all valid cases ────────────────────────────────────
results = []
for idx, row in df_valid.iterrows():
    cid = row["case_id"]
    cpath = row["case_path"]
    print(f"[{idx+1}/{len(df_valid)}] Processing {cid} …", end=" ")
    try:
        info = preprocess_single_case(cpath, cid, OUTPUT_DIR)
        results.append(info)
        print(f"✓  {info['orig_size']} → {info['proc_size']}  "
              f"body={info['body_voxel_pct']}%  bone={info['bone_voxel_pct']}%")
    except Exception as e:
        print(f"✗  ERROR: {e}")

df_results = pd.DataFrame(results)
print(f"\nDone. {len(df_results)} volumes saved to {OUTPUT_DIR}")
display(df_results)

## 10. Verification & Visualization

In [ ]:
# ── Verification checks ─────────────────────────────────────────
print("=" * 60)
print("VERIFICATION")
print("=" * 60)

all_ok = True
for _, row in df_results.iterrows():
    cid = row["case_id"]

    # Check volume
    fpath = row["vol_path"]
    img = sitk.ReadImage(fpath)
    sz  = img.GetSize()
    sp  = tuple(round(s, 4) for s in img.GetSpacing())
    arr = sitk.GetArrayFromImage(img)
    imin, imax = float(arr.min()), float(arr.max())

    checks = []
    if sz != TARGET_SHAPE[::-1]:   # sitk uses (X,Y,Z)
        checks.append(f"vol shape {sz} ≠ {TARGET_SHAPE[::-1]}")
    if sp != TARGET_SPACING:
        checks.append(f"vol spacing {sp} ≠ {TARGET_SPACING}")
    if imin < -0.001 or imax > 1.001:
        checks.append(f"vol intensity [{imin:.4f}, {imax:.4f}] outside [0,1]")

    # Check bone mask
    bpath = row["bone_path"]
    bimg = sitk.ReadImage(bpath)
    bsz  = bimg.GetSize()
    barr = sitk.GetArrayFromImage(bimg)
    buniq = set(np.unique(barr))

    if bsz != TARGET_SHAPE[::-1]:
        checks.append(f"bone shape {bsz} ≠ {TARGET_SHAPE[::-1]}")
    if not buniq.issubset({0, 1}):
        checks.append(f"bone mask not binary: {buniq}")

    status = "✓" if not checks else "✗ " + "; ".join(checks)
    if checks:
        all_ok = False
    print(f"  {cid:10s}  {status}")

print()
if all_ok:
    print("All checks PASSED ✓")
else:
    print("Some checks FAILED ✗ — inspect the output above.")

### Before vs. After Visualization

In [ ]:
# ── Show before / after for the first two valid cases ──────────
n_show = min(2, len(df_results))

for i in range(n_show):
    cid   = df_results.iloc[i]["case_id"]
    cpath = df_valid[df_valid["case_id"] == cid].iloc[0]["case_path"]

    # Load original
    raw_img   = load_dicom_volume(cpath)
    raw_arr   = sitk.GetArrayFromImage(raw_img)
    raw_mid_z = raw_arr.shape[0] // 2

    # Load processed
    proc_arr = sitk.GetArrayFromImage(sitk.ReadImage(df_results.iloc[i]["vol_path"]))
    bone_arr = sitk.GetArrayFromImage(sitk.ReadImage(df_results.iloc[i]["bone_path"]))
    proc_mid_z = proc_arr.shape[0] // 2

    fig, axes = plt.subplots(2, 3, figsize=(18, 12))

    # Row 1: Original
    axes[0, 0].imshow(raw_arr[raw_mid_z], cmap="gray")
    axes[0, 0].set_title(f"Original – Axial [{raw_arr.shape}]")
    axes[0, 0].axis("off")

    axes[0, 1].imshow(raw_arr[:, raw_arr.shape[1]//2, :], cmap="gray")
    axes[0, 1].set_title("Original – Coronal")
    axes[0, 1].axis("off")

    axes[0, 2].imshow(raw_arr[:, :, raw_arr.shape[2]//2], cmap="gray")
    axes[0, 2].set_title("Original – Sagittal")
    axes[0, 2].axis("off")

    # Row 2: Processed + Bone overlay
    axes[1, 0].imshow(proc_arr[proc_mid_z], cmap="gray", vmin=0, vmax=1)
    axes[1, 0].imshow(bone_arr[proc_mid_z], cmap="Reds", alpha=0.35)
    axes[1, 0].set_title(f"Processed + Bone – Axial [{proc_arr.shape}]")
    axes[1, 0].axis("off")

    axes[1, 1].imshow(proc_arr[:, proc_arr.shape[1]//2, :], cmap="gray", vmin=0, vmax=1)
    axes[1, 1].imshow(bone_arr[:, bone_arr.shape[1]//2, :], cmap="Reds", alpha=0.35)
    axes[1, 1].set_title("Processed + Bone – Coronal")
    axes[1, 1].axis("off")

    axes[1, 2].imshow(proc_arr[:, :, proc_arr.shape[2]//2], cmap="gray", vmin=0, vmax=1)
    axes[1, 2].imshow(bone_arr[:, :, bone_arr.shape[2]//2], cmap="Reds", alpha=0.35)
    axes[1, 2].set_title("Processed + Bone – Sagittal")
    axes[1, 2].axis("off")

    plt.suptitle(f"Before → After: {cid}", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()